# Notebook 10: Gradient Checkpointing

## Why This Matters

Backpropagation requires storing all intermediate activations from the forward pass. For a 7B model with a 4096-length sequence, this can be 100+ GB -- more than most GPU clusters have. Gradient checkpointing (activation recomputation) is the technique that makes this tractable: store only a fraction of activations and recompute the rest during the backward pass. This is a mandatory technique for training any large model, and understanding the memory-compute tradeoff is essential for ML infrastructure work.

In [ ]:
%pip install -q torch numpy matplotlib

In [ ]:
import torch
import torch.nn as nn
import torch.utils.checkpoint as checkpoint
import numpy as np
import matplotlib.pyplot as plt
import time
import gc

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")
print(f"PyTorch version: {torch.__version__}")
print("Ready.")

## 1. Why Activations Dominate Memory in Training

During a forward pass, PyTorch keeps all intermediate tensors alive for the backward pass. For a transformer:

- Each attention layer stores: Q, K, V projections, attention weights (seq² per head), attention output
- Each FFN layer stores: input, hidden activations, output
- For a GPT-3 scale model (96 layers, d=12288, seq=2048, batch=32):

$$\text{Activation memory} \approx \text{batch} \times \text{seq} \times d_{model} \times \text{bytes per element} \times \text{multiplier per layer} \times n_{layers}$$

Without gradient checkpointing: ~1 TB for GPT-3.
With full gradient checkpointing (checkpoint every layer): ~4 GB.
Cost: each layer's activations are recomputed during backward, adding ~33% compute overhead.

In [ ]:
# Measure activation memory with and without gradient checkpointing

class Layer(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.linear1 = nn.Linear(d, 4*d)
        self.linear2 = nn.Linear(4*d, d)
        self.norm = nn.LayerNorm(d)
    
    def forward(self, x):
        h = torch.relu(self.linear1(x))
        return self.norm(x + self.linear2(h))

class DeepModel(nn.Module):
    def __init__(self, d=256, n_layers=32, use_checkpoint=False):
        super().__init__()
        self.layers = nn.ModuleList([Layer(d) for _ in range(n_layers)])
        self.use_checkpoint = use_checkpoint
    
    def forward(self, x):
        for layer in self.layers:
            if self.use_checkpoint:
                x = checkpoint.checkpoint(layer, x, use_reentrant=False)
            else:
                x = layer(x)
        return x.mean()

def measure_memory_and_time(use_checkpoint, d=256, n_layers=16, batch=8, seq=512):
    model = DeepModel(d=d, n_layers=n_layers, use_checkpoint=use_checkpoint).to(device)
    x = torch.randn(batch, seq, d, device=device, requires_grad=True)
    
    if device == 'cuda':
        torch.cuda.reset_peak_memory_stats()
        torch.cuda.empty_cache()
    
    # Forward + backward
    times = []
    for _ in range(3):
        if device == 'cuda':
            torch.cuda.reset_peak_memory_stats()
        
        t0 = time.perf_counter()
        loss = model(x)
        loss.backward()
        t1 = time.perf_counter()
        
        times.append(t1 - t0)
        model.zero_grad()
        if x.grad is not None:
            x.grad.zero_()
    
    peak_mem_mb = torch.cuda.max_memory_allocated() / 1e6 if device == 'cuda' else 0
    return np.mean(times) * 1000, peak_mem_mb

print("Comparing standard vs gradient checkpointing:")
print(f"{'Mode':<30} {'Time (ms)':<15} {'Peak GPU memory (MB)'}")
print("-" * 60)

for use_ckpt, name in [(False, "Standard (no checkpoint)"), (True, "Gradient checkpoint")]:
    t_ms, mem_mb = measure_memory_and_time(use_ckpt)
    print(f"{name:<30} {t_ms:<15.1f} {mem_mb:<15.1f}")

print()
print("Memory savings come at ~33% compute overhead (each layer's forward is run twice).")
print("For CPU: memory effect less visible; time overhead is still measurable.")

## 2. How `torch.utils.checkpoint.checkpoint` Works

`torch.utils.checkpoint.checkpoint(function, *inputs)` does:

1. **Forward pass**: runs `function(*inputs)` but **does not save intermediate activations**. Only saves the inputs to the checkpointed segment.
2. **During backward**: when gradients need to flow through this segment, **re-runs the forward pass** on the saved inputs to reconstruct the intermediate activations.
3. The gradients are then computed using these reconstructed activations.

This means each checkpointed segment's activations are stored 0 times normally and recomputed once during backward. The peak memory at any point is: activations of 1 forward pass + gradients for the current segment.

In [ ]:
# Demonstrate the recomputation mechanism
# A custom checkpoint that shows when forward is called

class TrackedLayer(nn.Module):
    def __init__(self, layer_id, d=64):
        super().__init__()
        self.layer_id = layer_id
        self.linear = nn.Linear(d, d)
        self.forward_count = 0
    
    def forward(self, x):
        self.forward_count += 1
        call_type = "RECOMPUTE" if torch.is_grad_enabled() is False else "forward"
        # Note: during checkpoint recomputation, grad is disabled temporarily
        return torch.relu(self.linear(x))

# Build a 4-layer model with checkpointing
d = 64
layers = [TrackedLayer(i, d) for i in range(4)]
all_params = []
for l in layers:
    all_params.extend(l.parameters())
optimizer = torch.optim.SGD(all_params, lr=0.01)

x = torch.randn(2, d)

# With checkpointing
print("Forward pass with gradient checkpointing (4 layers):")
out = x
for layer in layers:
    out = checkpoint.checkpoint(layer, out, use_reentrant=False)

loss = out.sum()
print(f"  Forward calls after forward pass: {[l.forward_count for l in layers]}")

print("  Running backward (triggers recomputation)...")
loss.backward()
print(f"  Forward calls after backward pass: {[l.forward_count for l in layers]}")
print()
print("Observation: each layer's forward is called TWICE (once for forward, once for recompute).")
print("Without checkpointing: only called once.")

## 3. Granularity: What to Checkpoint

You can checkpoint at different granularities:

| Strategy | Memory | Compute overhead | Implementation |
|----------|--------|-----------------|----------------|
| **No checkpointing** | O(n_layers) | 0% | Standard training |
| **Checkpoint every layer** | O(1) per layer | ~33% | `checkpoint` each `TransformerLayer` |
| **Checkpoint every k layers** | O(k) | ~33%/k overhead | Checkpoint every kth layer |
| **Selective checkpointing** | O(expensive ops) | Minimal | Only checkpoint attention, not FFN |
| **Offload to CPU** | Near zero GPU | 5-10x slowdown | Move tensors to CPU RAM |

**Selective checkpointing**: the attention's softmax output (seq × seq × heads × batch) is the biggest single activation. Checkpointing just this drops most activation memory with minimal extra compute.

In [ ]:
# Selective checkpointing: only checkpoint expensive operations

class AttentionWithSelectiveCheckpoint(nn.Module):
    def __init__(self, d_model, n_heads, checkpoint_attn=True):
        super().__init__()
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        self.checkpoint_attn = checkpoint_attn
        
        self.qkv = nn.Linear(d_model, 3 * d_model)
        self.out = nn.Linear(d_model, d_model)
    
    def _attn_forward(self, q, k, v):
        """The expensive part: O(seq^2) attention weights."""
        scale = self.d_head ** -0.5
        scores = q @ k.transpose(-2, -1) * scale
        attn = torch.softmax(scores, dim=-1)
        return attn @ v
    
    def forward(self, x):
        B, S, D = x.shape
        qkv = self.qkv(x)
        q, k, v = qkv.chunk(3, dim=-1)
        
        q = q.view(B, S, self.n_heads, self.d_head).transpose(1, 2)
        k = k.view(B, S, self.n_heads, self.d_head).transpose(1, 2)
        v = v.view(B, S, self.n_heads, self.d_head).transpose(1, 2)
        
        if self.checkpoint_attn:
            # Only checkpoint the O(seq^2) part
            attn_out = checkpoint.checkpoint(self._attn_forward, q, k, v, use_reentrant=False)
        else:
            attn_out = self._attn_forward(q, k, v)
        
        out = attn_out.transpose(1, 2).contiguous().view(B, S, D)
        return self.out(out)

# Test both versions
torch.manual_seed(42)
d_model, n_heads, batch, seq = 128, 4, 2, 64

attn_standard = AttentionWithSelectiveCheckpoint(d_model, n_heads, checkpoint_attn=False)
attn_ckpt = AttentionWithSelectiveCheckpoint(d_model, n_heads, checkpoint_attn=True)

# Copy weights
attn_ckpt.load_state_dict(attn_standard.state_dict())

x = torch.randn(batch, seq, d_model)

out_std = attn_standard(x)
out_ckpt = attn_ckpt(x)

print(f"Output match (standard vs selective checkpoint): {torch.allclose(out_std, out_ckpt, atol=1e-5)}")
print()

# Memory estimate for attention weights
attn_weight_bytes = batch * n_heads * seq * seq * 4  # float32
print(f"Attention weight tensor: {attn_weight_bytes / 1e6:.1f} MB")
print(f"  For seq=2048: {batch * n_heads * 2048 * 2048 * 4 / 1e6:.0f} MB")
print(f"  For seq=8192: {batch * n_heads * 8192 * 8192 * 4 / 1e6:.0f} MB")
print()
print("This is why FlashAttention (fused attention kernel) is so important:")
print("It never materializes the full seq×seq attention matrix in HBM.")

## 4. Memory-Compute Tradeoff Analysis

The classic memory-compute tradeoff for gradient checkpointing:

With N total layers and checkpointing every k layers (k=1 means every layer):
- **Memory**: O(k) layers of activations in memory at any time
- **Extra compute**: one additional forward pass per checkpointed segment
- **Overhead**: ≈ 1/3 extra compute (one extra forward for every 2 forward+backward)

For k=sqrt(N) ("optimal checkpointing"): memory = O(sqrt(N)), compute overhead = O(sqrt(N) extra forward passes).

In [ ]:
# Memory-compute tradeoff visualization

def activation_memory_mb(n_layers, k, batch, seq, d_model, dtype_bytes=2):
    """
    Activation memory with checkpointing every k layers.
    At any point, we store activations for at most k layers.
    """
    per_layer_mb = batch * seq * d_model * dtype_bytes / 1e6
    return k * per_layer_mb

def compute_overhead_fraction(n_layers, k):
    """
    Fraction of extra compute.
    Each checkpoint segment of k layers does 2x forward (once for fwd, once for recompute).
    Total forward passes = n_layers + n_layers/k * k = 2*n_layers.
    But we only have 1 backward. So overhead vs no-checkpointing = n_layers/(n_layers + n_layers) = 1/3.
    More precisely, with k-layer segments:
    Standard: N forward + N backward = 3N total ops
    Checkpointed: N forward + N/k recompute + N backward = (3 + 1/k*? ) N total ops
    Simplified: overhead ≈ 1/3 regardless of k (for k < N)
    """
    if k >= n_layers:
        return 0.0
    return 1/3  # approximately

n_layers = 96
batch, seq, d_model = 4, 2048, 8192

k_values = list(range(1, 33))
memories = [activation_memory_mb(n_layers, k, batch, seq, d_model) for k in k_values]
overheads = [compute_overhead_fraction(n_layers, k) * 100 for k in k_values]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

ax1.plot(k_values, [m/1e3 for m in memories], 'b-o', markersize=4)
ax1.set_xlabel('Checkpoint interval k (layers)')
ax1.set_ylabel('Activation memory (GB)')
ax1.set_title('Activation memory vs checkpoint interval')
ax1.axhline(y=80, color='r', linestyle='--', label='A100 80GB')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(k_values, overheads, 'r-o', markersize=4)
ax2.set_xlabel('Checkpoint interval k (layers)')
ax2.set_ylabel('Extra compute overhead (%)')
ax2.set_title('Compute overhead vs checkpoint interval')
ax2.set_ylim(0, 50)
ax2.grid(True, alpha=0.3)

plt.suptitle(f'Gradient Checkpointing Tradeoff
(n_layers={n_layers}, batch={batch}, seq={seq}, d={d_model})',
             fontsize=11)
plt.tight_layout()
plt.savefig('checkpoint_tradeoff.png', dpi=100)
plt.show()

print(f"No checkpointing (k={n_layers}): {activation_memory_mb(n_layers, n_layers, batch, seq, d_model)/1e3:.0f} GB, 0% overhead")
print(f"Full checkpointing (k=1):  {activation_memory_mb(n_layers, 1, batch, seq, d_model)/1e3:.2f} GB, ~33% overhead")
print(f"Optimal (k=sqrt(N)={int(n_layers**0.5)}): {activation_memory_mb(n_layers, int(n_layers**0.5), batch, seq, d_model)/1e3:.1f} GB")

## Summary

### Key Concepts

| Strategy | Peak Memory | Compute Overhead | When to use |
|----------|-------------|-----------------|-------------|
| No checkpointing | O(n_layers) | 0% | Small models, ample memory |
| Checkpoint every k layers | O(k) | ~33% | Standard large model training |
| Checkpoint every layer (k=1) | O(1) | ~33% | Very large models on tight memory |
| Selective (attn only) | O(n_layers) - seq² | Minimal | When attention is bottleneck |
| CPU offload | ~0 GPU | 5-10x | Extreme memory constraints |

**Key facts**:
- `torch.utils.checkpoint.checkpoint` is 3 lines of code change
- Overhead is approximately 33% regardless of k (one extra forward pass per backward)
- FlashAttention avoids materializing the seq×seq attention matrix entirely -- complementary to gradient checkpointing
- For LLM fine-tuning: gradient checkpointing + LoRA is the standard recipe for training on limited hardware

**Next**: `11_profiling_distributed_training.ipynb` -- finding the actual bottleneck in your distributed training job.